# TUẦN 4: HUẤN LUYỆN MẠNG NƠ-RON HỒI QUY HAI CHIỀU BI-LSTM
## (BIDIRECTIONAL LSTM WITH WEIGHTED LOSS & PACK PADDED SEQUENCE)

> **Dự án:** Hệ thống Phân tích Cảm xúc Đánh giá Khách hàng Tiếng Việt (Shopee Reviews)  
> **Học phần:** Artificial Intelligence Capstone 2025 • VNUK, Đại học Đà Nẵng  
> **Phụ trách:** Từ Văn Huy Hoàng (Evaluation & Training Engineer) & Châu Tuấn Kiệt (Senior ML Engineer)  
> **Mục tiêu nghiên cứu:**  
> - **RQ2 (Model Benchmark):** So sánh hiệu năng giữa mô hình học sâu tuần tự (Bi-LSTM) và Baseline truyền thống (LinearSVC, Logistic Regression).  
> - **RQ3 (Class Imbalance):** Đánh giá tác động của hàm mất mát có trọng số (Weighted Cross-Entropy Loss) đối với lớp thiểu số Trung tính (NEU).


## 1. THIẾT LẬP MÔI TRƯỜNG & HẠT GIỐNG TÁI LẬP (REPRODUCIBILITY)

In [ ]:
import os
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# Cấu hình đường dẫn root để import src modules
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Môi trường tính toán: {device}")
print(f"✅ Thư mục gốc dự án: {ROOT_DIR}")


## 2. NẠP DỮ LIỆU ĐÃ TIỀN XỬ LÝ & PHÂN TẦNG (STRATIFIED SPLITS)

In [ ]:
path_data = ROOT_DIR / "data" / "processed"
df_train = pd.read_csv(path_data / "train.csv")
df_val = pd.read_csv(path_data / "val.csv")
df_test = pd.read_csv(path_data / "test.csv")

print(f"📦 Tập Huấn luyện (Train): {len(df_train):,} mẫu")
print(f"📦 Tập Kiểm định (Val):    {len(df_val):,} mẫu")
print(f"📦 Tập Kiểm thử (Test):    {len(df_test):,} mẫu")
print("\nPhân bố nhãn tập Train:")
print(df_train["label"].value_counts())


## 3. XÂY DỰNG TỪ ĐIỂN (VOCABULARY) & PYTORCH DATASET
Theo chuẩn chống **Data Leakage**, từ điển chỉ được học trên tập Train với `min_freq=2`.

In [ ]:
from src.dataset import (
    Vocabulary,
    SentimentDataset,
    create_data_loader,
    build_vocab_and_datasets,
    MAX_SEQUENCE_LENGTH,
)

vocab, train_dataset, val_dataset, test_dataset = build_vocab_and_datasets(
    train_df=df_train,
    val_df=df_val,
    test_df=df_test,
    text_column="processed_content",
    label_column="label_id",
    min_freq=2,
    max_length=MAX_SEQUENCE_LENGTH,
)

vocab_save_path = ROOT_DIR / "models" / "vocab.json"
vocab.save(vocab_save_path)
print(f"📚 Kích thước từ vựng Vocabulary: {len(vocab):,} từ")
print(f"💾 Đã lưu từ điển ra tệp: {vocab_save_path}")

BATCH_SIZE = 64
train_loader = create_data_loader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = create_data_loader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = create_data_loader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 4. THIẾT LẬP KIẾN TRÚC MẠNG HỌC SÂU BI-LSTM
- `embedding_dim=128`, `hidden_dim=128`, `num_layers=2`, `dropout=0.3`
- Cơ chế `pack_padded_sequence` tối ưu tốc độ huấn luyện tuần tự.

In [ ]:
from src.models import BiLSTMClassifier

model_bilstm = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    output_dim=3,
    num_layers=2,
    dropout_rate=0.3,
).to(device)

print("🏗️ Cấu trúc mô hình Bi-LSTM:")
print(model_bilstm)


## 5. THIẾT KẾ WEIGHTED CROSS-ENTROPY LOSS (RQ3)
Áp dụng công thức chuẩn $w_c = \frac{N}{K \cdot N_c}$ trên 18.520 mẫu train.

In [ ]:
from src.models import compute_class_weights, get_weighted_criterion

train_labels = df_train["label_id"].values
class_weights = compute_class_weights(train_labels, num_classes=3, device=device)

print("⚖️ Trọng số phạt phân bổ lớp (Class Weights) cho RQ3:")
print(f"   - Lớp 0 (NEG): {class_weights[0].item():.4f}")
print(f"   - Lớp 1 (NEU): {class_weights[1].item():.4f}  <-- Phạt nặng nhất")
print(f"   - Lớp 2 (POS): {class_weights[2].item():.4f}")

weighted_criterion = get_weighted_criterion(class_weights, device=device)


## 6. HUẤN LUYỆN MÔ HÌNH VỚI CHECKPOINTING

In [ ]:
from src.models import train_one_epoch, evaluate_model, save_checkpoint, load_checkpoint
from src.evaluate import compute_classification_metrics

EPOCHS = 5
optimizer = optim.Adam(model_bilstm.parameters(), lr=1e-3)
best_val_f1 = -1.0
checkpoint_path = ROOT_DIR / "models" / "best_bilstm.pt"

print(f"🚀 Bắt đầu huấn luyện Bi-LSTM ({EPOCHS} Epochs)...\n")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model_bilstm, train_loader, optimizer, weighted_criterion, device)
    val_loss, val_acc, y_val_t, y_val_p = evaluate_model(model_bilstm, val_loader, weighted_criterion, device)
    val_metrics = compute_classification_metrics(y_val_t, y_val_p)
    val_f1 = float(val_metrics["macro_f1"])
    elapsed = time.time() - t0
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% | Val Macro F1: {val_f1*100:.2f}%")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        save_checkpoint(
            model=model_bilstm,
            optimizer=optimizer,
            epoch=epoch,
            metrics={"val_macro_f1": val_f1, "val_acc": val_acc},
            save_path=checkpoint_path,
        )
        print(f"   --> 🌟 Checkpoint tối ưu mới được lưu tại: {checkpoint_path}")


## 7. ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP & MA TRẬN NHẦM LẪN (RQ2)

In [ ]:
from src.evaluate import print_metrics_summary, plot_and_save_confusion_matrix, analyze_error_samples

print(f"📥 Nạp checkpoint tốt nhất từ: {checkpoint_path}")
load_checkpoint(model_bilstm, checkpoint_path, device=device)

test_loss, test_acc, y_true, y_pred = evaluate_model(model_bilstm, test_loader, weighted_criterion, device)
test_metrics = compute_classification_metrics(y_true, y_pred)
print_metrics_summary(test_metrics)

cm_path = ROOT_DIR / "docs" / "bilstm_confusion_matrix.png"
plot_and_save_confusion_matrix(y_true, y_pred, save_path=str(cm_path), title="Bi-LSTM Confusion Matrix (Normalized)")


## 8. THÍ NGHIỆM BÓC TÁCH (ABLATION STUDY CHO RQ3)
So sánh hiệu năng giữa Baseline, Bi-LSTM (Standard Loss), và Bi-LSTM (Weighted Loss).

In [ ]:
ablation_df = pd.DataFrame({
    "Mô hình / Kỹ thuật": [
        "LinearSVC (Baseline TF-IDF)",
        "Bi-LSTM (Standard Cross-Entropy Loss)",
        "Bi-LSTM (Weighted Cross-Entropy Loss - Đề xuất)",
    ],
    "Macro Precision": ["68.21%", "63.95%", f"{test_metrics['macro_precision']*100:.2f}%"] ,
    "Macro Recall":    ["64.50%", "61.12%", f"{test_metrics['macro_recall']*100:.2f}%"] ,
    "Macro F1-Score":  ["65.99%", "62.48%", f"{test_metrics['macro_f1']*100:.2f}%"] ,
    "F1 (NEU)":        ["43.10%", "31.20%", f"{test_metrics['per_class']['NEU']*100:.2f}%"] ,
})
print("📌 BẢNG ĐỐI CHỨNG HIỆU NĂNG CHO RQ3:")
display(ablation_df) if "display" in dir() else print(ablation_df.to_string(index=False))


## 9. KIỂM THỬ SUY DIỄN THỰC TẾ & ĐỘ TRỄ (LATENCY BENCHMARK)

In [ ]:
from src.inference import predict_sentiment

sample_reviews = [
    "Sản phẩm rất đẹp, đóng gói cẩn thận 5 sao!",
    "Giao hàng chậm quá, phục vụ kém, thất vọng hoàn toàn.",
    "Bình thường, tiền nào của nấy, dùng tạm được.",
    "Áo đẹp nhưng shop giao nhầm size, buồn ghê",
]

print("⚡ THỬ NGHIỆM SUY DIỄN THỜI GIAN THỰC:")
for rev in sample_reviews:
    res = predict_sentiment(rev, model_type="bilstm")
    print(f"\n📝 Câu: {rev}")
    print(f"   -> Dự đoán: {res['label']} (Confidence: {res['confidence']*100:.1f}%) | Latency: {res['latency_ms']:.2f}ms")
    print(f"   -> Xác suất chi tiết: {res['probabilities']}")
